In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine
import os

def connect_to_db():
    engine = create_engine(
        f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
        f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    )
    return engine

engine = connect_to_db()

conn = engine.connect()

In [ ]:
df = pd.read_sql("SELECT * FROM public.gold_model_dataset", engine)

df.head()

In [ ]:
df["is_booking"].mean()

In [ ]:
def plot_categorical_vs_target(col):
    temp = df.groupby(col)["is_booking"].mean().sort_values(ascending=False)

    temp.plot(kind="bar", figsize=(10,4))
    plt.title(f"{col} vs Booking Rate")
    plt.ylabel("Booking Rate")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plot_categorical_vs_target("is_mobile")
plot_categorical_vs_target("channel")
plot_categorical_vs_target("trip_type")
plot_categorical_vs_target("is_package")

In [ ]:
def plot_numeric_vs_target(col, bins=10):
    temp = df[[col, "is_booking"]].copy()
    temp["bin"] = pd.qcut(temp[col], q=bins, duplicates="drop")

    grouped = temp.groupby("bin")["is_booking"].mean()

    grouped.plot(kind="bar", figsize=(10,4))
    plt.title(f"{col} vs Booking Rate")
    plt.ylabel("Booking Rate")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plot_numeric_vs_target("advance_booking_days")
plot_numeric_vs_target("stay_duration")
plot_numeric_vs_target("orig_destination_distance")
plot_numeric_vs_target("cnt")

In [ ]:
plot_numeric_vs_target("total_guests")

In [ ]:
df.groupby("is_mobile")["is_booking"].mean()

In [ ]:
df.groupby("is_package")["is_booking"].mean()

In [ ]:
df.groupby("trip_type")["is_booking"].mean()

In [ ]:
baseline = df["is_booking"].mean()
baseline

In [ ]:
def plot_with_lift(col):
    temp = df.groupby(col)["is_booking"].mean()
    lift = temp / baseline

    lift.plot(kind="bar", figsize=(10,4))
    plt.title(f"{col} - Lift vs Baseline")
    plt.ylabel("Lift")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plot_with_lift("is_mobile")
plot_with_lift("trip_type")
plot_with_lift("channel")

In [ ]:
def plot_numeric_full(col, bins=10):
    temp = df[[col, "is_booking"]].copy()
    temp["bin"] = pd.qcut(temp[col], q=bins, duplicates="drop")

    grouped = temp.groupby("bin").agg({
        "is_booking": "mean",
        col: "count"
    })

    fig, ax1 = plt.subplots(figsize=(10,4))

    grouped["is_booking"].plot(kind="bar", ax=ax1)
    ax1.set_ylabel("Booking Rate")

    ax2 = ax1.twinx()
    grouped[col].plot(color="red", ax=ax2)
    ax2.set_ylabel("Volume")

    plt.title(f"{col} vs Booking Rate + Volume")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plot_numeric_full("advance_booking_days")
plot_numeric_full("stay_duration")
plot_numeric_full("cnt")

In [ ]:
df.groupby(["is_mobile", "is_package"])["is_booking"].mean()

In [ ]:
top_channels = df["channel"].value_counts().head(10).index

df[df["channel"].isin(top_channels)] \
    .groupby("channel")["is_booking"].mean() \
    .sort_values(ascending=False)

In [ ]:
conn.close()

## Key Insights

- Device type has a strong impact on conversion, with mobile users showing significantly lower booking rates compared to desktop users.
- Package bookings are associated with higher conversion rates, indicating stronger purchase intent.
- Session intensity (`cnt`) reveals an inverse relationship with booking, suggesting that users with many interactions are more likely to be exploring rather than converting.
- Shorter booking windows tend to have higher conversion rates, indicating that last-minute decisions are more likely to result in bookings.
- Distance plays a meaningful role in user behavior, with nearby destinations showing higher conversion rates, while long-distance searches convert less frequently.
- Users with unknown distance values exhibit distinct behavior, reinforcing the importance of treating missing data as a separate category.

## Conclusion

The analysis highlights that booking behavior is strongly influenced by a combination of user context (device, package), behavioral signals (session intensity), and trip characteristics (distance and timing). These patterns provide clear guidance for both modeling and business decision-making, as they capture how different user segments interact with the platform.